# Tanshi — Production Avatar Training — Cloud Environment (Phase B0)

Prepares a **complete, resumable** training environment for a talking-head avatar
from the production avatar dataset (461 accepted clips, 61 min, readiness 95/100).

**Supported models:** `musetalk` · `latentsync` · `echomimic` · `hallo2`

**What it does:** mounts Drive → clones the repo → installs dependencies →
detects GPU/CUDA → locates + integrity-verifies the dataset (sha256 manifest) →
stages it to fast local disk → prepares checkpoint/resume + Drive-backed
logs/metrics → builds the exact training command.

**It does NOT auto-start training** — the launch cell is guarded by
`START_TRAINING = False` (Phase B0 prepares only).

**Before first run (one-time):**
1. Upload `avatar_dataset` (the folder itself, structure preserved) anywhere under
   `MyDrive` — see `docs/AVATAR_DRIVE_UPLOAD.md` in the repo.
2. Runtime → Change runtime type → **GPU** (A100/L4/T4; more VRAM = larger batches).
3. Edit the CONFIG cell, then Run All.

In [ ]:
# ============================ CONFIG — the only cell you edit ============================
MODEL = "musetalk"            #@param ["musetalk", "latentsync", "echomimic", "hallo2"]

REPO_URL = "https://github.com/nahatadhananjay33-svg/ai-creator-platform.git"
BRANCH   = "main"           # cloud setup is merged into main

DATASET_DIRNAME = "avatar_dataset"      # folder name to auto-discover under MyDrive
# uploaded 2026-07-17 to MyDrive/Ai_creator/Digital_Avatar_Tanshi/avatar_dataset
# (folder id 1JGyi0atGEA79-2v2l3SMSFNHMuG8Fw7f); clear this to force discovery
DATASET_PATH    = "/content/drive/MyDrive/Ai_creator/Digital_Avatar_Tanshi/avatar_dataset"
WORK_DIRNAME    = "avatar_training"     # Drive folder for checkpoints/logs/metrics

FULL_HASH_VERIFY = False                # True = re-hash all 16.7 GB (slow but airtight)
STAGE_TO_LOCAL   = True                 # copy accepted clips to /content for fast I/O
START_TRAINING   = False                # Phase B0: leave False. Flip ONLY to launch.
print(f"model={MODEL}  branch={BRANCH}  verify={'full' if FULL_HASH_VERIFY else 'quick'}")

In [ ]:
# ============================ 1. Mount Google Drive ============================
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
MYDRIVE = Path("/content/drive/MyDrive")
WORK = MYDRIVE / WORK_DIRNAME
for sub in ("checkpoints", "logs", "metrics", "benchmarks"):
    (WORK / sub / MODEL).mkdir(parents=True, exist_ok=True)
print(f"Drive mounted. Work dir: {WORK}")

In [ ]:
# ============================ 2. GPU / CUDA detection ============================
import shutil, subprocess, sys

if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU runtime! Runtime -> Change runtime type -> GPU, then rerun.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
import torch
print(f"torch {torch.__version__}  cuda available: {torch.cuda.is_available()}"
      f"  device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a'}")
assert torch.cuda.is_available(), "CUDA not available - check the runtime type"

In [ ]:
# ============================ 3. Clone repo + base dependencies ============================
import os, subprocess
from pathlib import Path

REPO = Path("/content/ai_creator_platform")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)],
                   check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "opencv-python-headless", "openpyxl", "tqdm"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# ============================ 4. Locate the dataset on Drive ============================
DATASET_FOLDER_ID = "1JGyi0atGEA79-2v2l3SMSFNHMuG8Fw7f"   # Drive id of avatar_dataset

def find_dataset(mydrive: Path, dirname: str) -> Path:
    candidates = []
    if DATASET_PATH:
        candidates.append(Path(DATASET_PATH))
    # a My Drive SHORTCUT to a shared folder surfaces here in the Colab mount
    candidates.append(Path("/content/drive/.shortcut-targets-by-id")
                      / DATASET_FOLDER_ID / dirname)
    for p in candidates:
        if (p / "dataset.sqlite").exists():
            return p
    print("note: direct paths missing - walking MyDrive (can take a minute)...")
    hits = [Path(r) for r, dirs, files in os.walk(mydrive)
            if Path(r).name == dirname and "dataset.sqlite" in files]
    if len(hits) > 1:
        print(f"note: {len(hits)} candidates, using the first: {hits[0]}")
    if hits:
        return hits[0]
    raise SystemExit(
        f"'{dirname}' not reachable in this mount.\n"
        "The dataset lives in nahatadhananjay33@gmail.com's My Drive. If Drive shows\n"
        "it under 'Shared with me', you mounted a DIFFERENT account. Fix either way:\n"
        "  a) Runtime -> Disconnect and delete runtime, rerun, and when the Drive\n"
        "     popup asks which account to authorize, pick nahatadhananjay33@gmail.com; or\n"
        "  b) in Drive (this account), right-click avatar_dataset under Shared with me\n"
        "     -> Organise -> Add shortcut -> My Drive, then rerun this cell.")

DATASET = find_dataset(MYDRIVE, DATASET_DIRNAME)
print(f"Dataset: {DATASET}")

In [ ]:
# ============================ 5. Verify upload integrity (sha256 manifest) ============================
from production.cloud_setup.manifest import load_manifest, verify_against

man = load_manifest("production/cloud_setup/manifest/avatar_dataset_manifest.json")
print(f"manifest: {man['file_count']} files, {man['total_gb']} GB, "
      f"scope: {man['upload_scope']}")
res = verify_against(man, DATASET, quick=not FULL_HASH_VERIFY)
print(f"verified {res['verified']}/{res['expected_files']}  "
      f"missing {res['missing_count']}  size-bad {res['size_mismatch_count']}  "
      f"hash-bad {res['hash_mismatch_count']}")
assert res["ok"], f"UPLOAD INCOMPLETE/CORRUPT - first problems: " \
                  f"{(res['missing'] + res['size_mismatch'] + res['hash_mismatch'])[:5]}"
print("Upload integrity: PASS")

In [ ]:
# ============================ 6. Stage dataset to fast local disk ============================
# Drive's FUSE mount drops under sustained bulk reads (OSError errno 107,
# "Transport endpoint is not connected") - every copy retries with a forced
# remount, the same pattern the voice pipeline uses. Resume-safe: rerunning
# skips clips already staged at the right size.
import shutil, time

def _remount():
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print(f"  remount attempt failed: {e}")

def robust_copy(src: Path, dst: Path, retries: int = 4) -> None:
    for attempt in range(1, retries + 1):
        try:
            shutil.copyfile(src, dst)
            return
        except OSError as e:
            if attempt == retries:
                raise
            print(f"  Drive hiccup (errno {e.errno}) on {src.name} - "
                  f"remounting, retry {attempt}/{retries - 1}")
            _remount()
            time.sleep(2.0 * attempt)

def robust_list(folder: Path, retries: int = 4):
    for attempt in range(1, retries + 1):
        try:
            return sorted(folder.glob("*"))
        except OSError:
            if attempt == retries:
                raise
            _remount()
            time.sleep(2.0 * attempt)

LOCAL_DATASET = Path("/content/avatar_dataset")
if STAGE_TO_LOCAL:
    (LOCAL_DATASET / "accepted").mkdir(parents=True, exist_ok=True)
    for name in ("dataset.sqlite", "dataset.csv", "dataset.xlsx"):
        if not (LOCAL_DATASET / name).exists():
            robust_copy(DATASET / name, LOCAL_DATASET / name)
    from tqdm.auto import tqdm
    clips = robust_list(DATASET / "accepted")
    for src in tqdm(clips, desc="staging", unit="clip"):
        dst = LOCAL_DATASET / "accepted" / src.name
        if not (dst.exists() and dst.stat().st_size == src.stat().st_size):  # resume
            robust_copy(src, dst)
    TRAIN_DATA = LOCAL_DATASET
else:
    TRAIN_DATA = DATASET
print(f"Training data root: {TRAIN_DATA} "
      f"({len(list((TRAIN_DATA / 'accepted').glob('*')))} clips)")

In [ ]:
# ============================ 7. Model registry + install selected model ============================
# Colab-safe install strategy: NEVER let a repo's requirements.txt replace
# Colab's own CUDA torch stack. Framework lines are filtered out; the rest
# installs in bulk; on failure each package retries individually, and a
# package whose exact PIN has no build for Colab's Python retries UNPINNED
# (latest compatible) - relaxed pins are listed at the end for review.
import re
import subprocess
import sys
from pathlib import Path

MODELS = {
    "musetalk": {
        "repo": "https://github.com/TMElyralab/MuseTalk.git",
        "install": ["REQUIREMENTS", "pip install --no-cache-dir -U openmim",
                     "mim install mmengine", "mim install 'mmcv>=2.0.1'",
                     "mim install 'mmdet>=3.1.0'", "mim install 'mmpose>=1.1.0'"],
        "weights": "sh ./download_weights.sh",
        "train": "python train.py --data_root {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume {latest}",
    },
    "latentsync": {
        "repo": "https://github.com/bytedance/LatentSync.git",
        "install": ["REQUIREMENTS"],
        "weights": "huggingface-cli download ByteDance/LatentSync-1.5 --local-dir checkpoints",
        "train": "python -m scripts.train_unet --config configs/unet/stage2.yaml "
                  "--data_dir {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume_from_checkpoint {latest}",
    },
    "echomimic": {
        "repo": "https://github.com/antgroup/echomimic.git",
        "install": ["REQUIREMENTS"],
        "weights": "git lfs install && git clone https://huggingface.co/BadToBest/EchoMimic pretrained_weights",
        "train": "accelerate launch train_stage1.py --config configs/train/stage1.yaml "
                  "--data_root {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume {latest}",
    },
    "hallo2": {
        "repo": "https://github.com/fudan-generative-vision/hallo2.git",
        "install": ["REQUIREMENTS", "pip install -e ."],
        "weights": "huggingface-cli download fudan-generative-ai/hallo2 --local-dir pretrained_models",
        "train": "accelerate launch scripts/train_stage1.py --config configs/train/stage1.yaml "
                  "--data_root {data} --output_dir {ckpt} {resume_flag}",
        "resume_flag": "--resume {latest}",
    },
}

# frameworks Colab already provides with matched CUDA builds - never reinstall
_SKIP = re.compile(r"^(torch|torchvision|torchaudio|tensorflow|tensorboard|jax|"
                   r"nvidia-|xformers|triton)\s*($|[=<>!~\[])", re.I)


def _pip(*args) -> bool:
    return subprocess.run([sys.executable, "-m", "pip", "install", *args]).returncode == 0


def install_requirements(repo_dir: Path):
    """Returns (relaxed_pins, failed). Skips Colab-provided frameworks."""
    req = repo_dir / "requirements.txt"
    if not req.exists():
        print("  (no requirements.txt)")
        return [], []
    keep, skipped = [], []
    for line in req.read_text().splitlines():
        s = line.split("#", 1)[0].strip()
        if not s:
            continue
        (skipped if _SKIP.match(s) else keep).append(s)
    if skipped:
        print(f"  keeping Colab's own versions of: {', '.join(skipped)}")
    colab_req = repo_dir / "requirements_colab.txt"
    colab_req.write_text("\n".join(keep))
    if _pip("-r", str(colab_req)):
        return [], []
    print("\n  bulk install failed - per-package pass (bad pins retry unpinned)...")
    relaxed, failed = [], []
    for pkg in keep:
        if _pip(pkg):
            continue
        name_only = re.split(r"[=<>!~;@]", pkg, 1)[0].strip()
        if name_only and name_only != pkg and _pip(name_only):
            relaxed.append(f"{pkg} -> {name_only} (latest)")
            print(f"  RELAXED: {pkg} -> latest {name_only}")
        else:
            failed.append(pkg)
            print(f"  FAILED : {pkg}")
    return relaxed, failed


cfg = MODELS[MODEL]
MODEL_DIR = Path(f"/content/{MODEL}")
if not MODEL_DIR.exists():
    subprocess.run(["git", "clone", cfg["repo"], str(MODEL_DIR)], check=True)
os.chdir(MODEL_DIR)

relaxed_pins, failed_steps = [], []
for step in cfg["install"]:
    if step == "REQUIREMENTS":
        r, f = install_requirements(MODEL_DIR)
        relaxed_pins += r
        failed_steps += [f"pip: {p}" for p in f]
    else:
        print(f"$ {step}")
        if subprocess.run(step, shell=True).returncode != 0:
            failed_steps.append(step)

import torch as _t
print(f"\ntorch still healthy: {_t.__version__}, cuda={_t.cuda.is_available()}")
if relaxed_pins:
    print(f"\nNOTE - {len(relaxed_pins)} pin(s) had no build for this Python and were "
          f"installed at their latest version instead (usually fine; if training\n"
          f"errors mention one of these, that is the first suspect):")
    for s in relaxed_pins:
        print(f"  - {s}")
if failed_steps:
    print(f"\nWARNING - {len(failed_steps)} install step(s) failed; review before a "
          f"long training run (often optional extras):")
    for s in failed_steps:
        print(f"  - {s}")
if not failed_steps:
    print(f"\n{MODEL} ready.")
print(f"Weights (run once, ~GBs):\n  $ {cfg['weights']}")

In [ ]:
# ============================ 8. Checkpoint / resume + Drive-backed logging ============================
import datetime, json

CKPT_DIR = WORK / "checkpoints" / MODEL          # persists across Colab sessions
LOG_DIR = WORK / "logs" / MODEL
METRICS_DIR = WORK / "metrics" / MODEL

def latest_checkpoint(ckpt_dir: Path):
    cands = [p for p in Path(ckpt_dir).rglob("*")
             if p.is_file() and p.suffix in (".pt", ".pth", ".ckpt", ".safetensors")]
    return max(cands, key=lambda p: p.stat().st_mtime) if cands else None

latest = latest_checkpoint(CKPT_DIR)
resume_flag = cfg["resume_flag"].format(latest=latest) if latest else ""
TRAIN_CMD = cfg["train"].format(data=TRAIN_DATA, ckpt=CKPT_DIR, resume_flag=resume_flag)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = LOG_DIR / f"train_{stamp}.log"

# environment snapshot for reproducibility
snap = {"model": MODEL, "branch": BRANCH, "timestamp": stamp,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "resume_from": str(latest) if latest else None, "train_cmd": TRAIN_CMD}
(METRICS_DIR / f"env_{stamp}.json").write_text(json.dumps(snap, indent=2))
subprocess.run(f"pip freeze > '{LOG_DIR / f'pip_freeze_{stamp}.txt'}'", shell=True)

print(f"Checkpoints : {CKPT_DIR}  (resume from: {latest or 'fresh start'})")
print(f"Log file    : {LOG_FILE}")
print(f"Train cmd   : {TRAIN_CMD}")

In [ ]:
# ============================ 9. Launch (guarded — Phase B0 does NOT train) ============================
if not START_TRAINING:
    print("START_TRAINING = False (Phase B0: environment prepared, training NOT started).")
    print("Everything below is ready:")
    print(f"  dataset  : {TRAIN_DATA} (verified against the sha256 manifest)")
    print(f"  model    : {MODEL} installed at {MODEL_DIR}")
    print(f"  command  : {TRAIN_CMD}")
    print("Flip START_TRAINING = True in CONFIG to launch a real run.")
else:
    # tee output to the Drive log so interrupted sessions keep their history;
    # checkpoints go straight to Drive, so a new session resumes automatically.
    subprocess.run(f"{TRAIN_CMD} 2>&1 | tee -a '{LOG_FILE}'", shell=True, check=True)

In [ ]:
# ============================ 10. Benchmarks / outputs to Drive (post-training) ============================
BENCH_DIR = WORK / "benchmarks" / MODEL
print(f"Benchmark outputs dir (Drive): {BENCH_DIR}")
print("After a training run completes, place inference samples + metric JSONs here;")
print("logs and environment snapshots from every session are already in:")
print(f"  {LOG_DIR}\n  {METRICS_DIR}")
print("\nEnvironment preparation complete - no training was started (Phase B0).")